<a href="https://colab.research.google.com/github/joyashre/LLM-NLP-Learning-task/blob/main/NLP_task_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 8

In [ ]:
# Cell 1: Load the Causal LM for Prompting
import torch
from transformers import pipeline

# Hum TinyLlama use kar rahe hain kyunki yeh ek lightweight aur smart Causal LM hai
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

print(f"Downloading and loading {model_id}...")
print("(Isme 1-2 minute lag sakte hain...)")

# Text-generation pipeline create kar rahe hain
llm_pipeline = pipeline(
    "text-generation",
    model=model_id,
    # Agar GPU available hai toh bfloat16 use karega (memory bachane ke liye)
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("Model loaded successfully!")

(Isme 1-2 minute lag sakte hain...)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Model loaded successfully!


## Zero-shot, Few-shot, aur Chain of Thought Prompting

In [ ]:
# Cell 2: Coreference Resolution with 3 Prompting Techniques

# Hamara tricky test sentence
sentence = "The doctor told the patient that he needs to rest for a week."
question = "Who does the pronoun 'he' refer to in the sentence?"

In [ ]:
# ---------------------------------------------------------
# 1. Zero-Shot Prompt (Direct Question)
# ---------------------------------------------------------
zero_shot_prompt = f"""<|system|>
You are an AI assistant. Answer short and to the point.</s>
<|user|>
Sentence: {sentence}
Question: {question}</s>
<|assistant|>
Answer:"""

In [ ]:
# ---------------------------------------------------------
# 2. Few-Shot Prompt (Question with an Example)
# ---------------------------------------------------------
few_shot_prompt = f"""<|system|>
You are an AI assistant. Answer short and to the point.</s>
<|user|>
Sentence: The dog barked at the tree because it saw a squirrel.
Question: What does 'it' refer to in the sentence?</s>
<|assistant|>
Answer: The dog.</s>
<|user|>
Sentence: {sentence}
Question: {question}</s>
<|assistant|>
Answer:"""

In [ ]:

# ---------------------------------------------------------
# 3. Chain of Thought (CoT) Prompt (Step-by-Step Reasoning)
# ---------------------------------------------------------
cot_prompt = f"""<|system|>
You are a helpful and logical AI assistant.</s>
<|user|>
Sentence: {sentence}
Question: {question}</s>
<|assistant|>
Let's think step by step:"""

In [ ]:
# Helper function to generate output
def get_model_response(prompt, max_tokens=20):
    outputs = llm_pipeline(
        prompt,
        max_new_tokens=max_tokens,
        temperature=0.1, # Keep it deterministic
        do_sample=True,
        pad_token_id=llm_pipeline.tokenizer.eos_token_id
    )
    # Extract only the newly generated text
    full_text = outputs[0]['generated_text']
    generated_answer = full_text[len(prompt):].strip()
    return generated_answer

In [ ]:
# Output Results
print(f"Test Sentence: '{sentence}'")
print("="*50)

print("1. ZERO-SHOT OUTPUT:")
print(get_model_response(zero_shot_prompt, max_tokens=10))
print("-" * 50)

print("2. FEW-SHOT OUTPUT:")
print(get_model_response(few_shot_prompt, max_tokens=10))
print("-" * 50)

print("3. CHAIN OF THOUGHT (CoT) OUTPUT:")
# CoT ke liye zyada tokens chahiye kyunki wo reasoning explain karega
print(get_model_response(cot_prompt, max_tokens=200))
print("="*50)

Both `max_new_tokens` (=10) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test Sentence: 'The doctor told the patient that he needs to rest for a week.'
1. ZERO-SHOT OUTPUT:


Both `max_new_tokens` (=10) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The pronoun 'he' refers to the patient
--------------------------------------------------
2. FEW-SHOT OUTPUT:


Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The doctor. The pronoun 'he' refers
--------------------------------------------------
3. CHAIN OF THOUGHT (CoT) OUTPUT:
1. The sentence "The doctor told the patient that he needs to rest for a week" contains two clauses:
   - The first clause is "The doctor told the patient that he needs to rest for a week."
   - The second clause is "He needs to rest for a week."

2. The first clause refers to the doctor, who is the subject of the sentence.

3. The second clause refers to the patient, who is the object of the sentence.

So, the pronoun "he" refers to the patient in the sentence.
